# 09 专家并行与上下文并行

## 简介

本笔记本介绍两种重要的并行策略：
- **专家并行（Expert Parallelism, EP）**: 用于 MoE（Mixture of Experts）模型，将 Expert 分布到不同 GPU
- **上下文并行（Context Parallelism, CP）**: 用于长序列场景，沿序列维度切分注意力计算

这两种策略在大模型时代非常重要——MoE 是 DeepSeek-V3/Mixtral 等模型的核心，
CP 是处理 128K+ 超长上下文的必要手段。

In [ ]:
import torch
import sys; sys.path.insert(0, '..')
from parallel.expert_parallel.expert_partition import partition_experts, get_expert_owner
from parallel.expert_parallel.token_dispatch import dispatch_tokens_to_experts
from parallel.context_parallel.ring_attention import ring_attention_step, rotate_kv
from parallel.context_parallel.sequence_partition import partition_sequence, create_cp_causal_mask
from parallel.context_parallel.cp_integration import analyze_cp_tp_memory, recommend_parallel_config

print("专家并行与上下文并行模块加载完成")

## 1. 专家并行（Expert Parallelism）

MoE 模型中，每个 Transformer 层的 FFN 被替换为多个 Expert（小型 FFN），
Router 根据 token 内容选择 top-k 个 Expert 进行计算。

专家并行将这些 Expert 分布到不同 GPU 上：

```
  GPU 0        GPU 1        GPU 2        GPU 3
  ──────       ──────       ──────       ──────
  Expert 0     Expert 1     Expert 2     Expert 3
  Expert 4     Expert 5     Expert 6     Expert 7

  (8 个 Expert 分配到 4 张 GPU, 每张 2 个 Expert)
```

**Token Dispatch 流程**:
1. Router 在每个 GPU 上本地计算 token 的 top-k Expert
2. All-to-All 通信：将 token 发送到持有对应 Expert 的 GPU
3. 各 GPU 用本地 Expert 计算
4. All-to-All 通信：将结果送回原 GPU

In [ ]:
# 专家分配演示
print("=" * 50)
print("专家并行 (Expert Parallelism) 演示")
print("=" * 50)

# 均匀分配
n_experts = 8
world_size = 4
print(f"\n{n_experts} 个 Expert 分配到 {world_size} 张 GPU:")
for rank in range(world_size):
    experts = partition_experts(n_experts, rank, world_size)
    print(f"  Rank {rank}: Expert {experts}")

# Expert 归属查询
print(f"\nExpert 归属映射 (get_expert_owner):")
for expert_idx in range(n_experts):
    owner = get_expert_owner(expert_idx, world_size)
    print(f"  Expert {expert_idx} → Rank {owner}")

# 非均匀分配 (模拟 DeepSeek-V3 风格, 部分 Expert 为共享 Expert)
print(f"\n非均匀分配: {n_experts + 1} 个 Expert (含 1 个共享), {world_size} GPU:")
for rank in range(world_size):
    experts = partition_experts(n_experts + 1, rank, world_size)
    print(f"  Rank {rank}: Expert {experts}")

# EP 通信开销
print(f"\nEP 通信分析:")
print(f"  每层 2 次 All-to-All (token dispatch + result return)")
print(f"  All-to-All 复杂度: O(N) 总数据量, O(P-1/P) 步完成")
print(f"  通信量 = token 数量 × 隐藏维度 × top_k")

In [ ]:
# Token Dispatch 演示
print("\n" + "=" * 50)
print("Token Dispatch 演示")
print("=" * 50)

# 模拟: batch=2, seq=4, dim=8, 4 个 expert, top_k=2
B, S, D = 2, 4, 8
x = torch.randn(B, S, D)

# Router 输出 top-2 expert indices
router_indices = torch.tensor([
    [[0, 2], [1, 3], [0, 1], [2, 3]],
    [[1, 0], [3, 2], [2, 1], [0, 3]],
])

n_experts_demo = 4
dispatched = dispatch_tokens_to_experts(x, router_indices, n_experts_demo)

print(f"输入形状: {list(x.shape)}")
print(f"Router top-k: {router_indices.shape[-1]}")
print(f"总 Expert 数: {n_experts_demo}")
print(f"\nToken 分发结果:")
for expert_idx, tokens in dispatched.items():
    print(f"  Expert {expert_idx}: {tokens.shape[0]} 个 token, 形状 {list(tokens.shape)}")

# 负载均衡分析
print(f"\n负载均衡分析:")
token_counts = {e: dispatched[e].shape[0] if e in dispatched else 0 for e in range(n_experts_demo)}
for e, cnt in token_counts.items():
    bar = "█" * cnt
    print(f"  Expert {e}: {bar} ({cnt} tokens)")

total_tokens = sum(token_counts.values())
print(f"  总计: {total_tokens} token-dispatch 次")
print(f"  理想每 Expert: {total_tokens / n_experts_demo:.1f} tokens")
print(f"  ★ 负载不均衡是 MoE 训练的主要挑战之一")
print(f"  ★ DeepSeek-V3 使用辅助损失 (auxiliary loss) 来鼓励负载均衡")

## 2. 上下文并行（Context Parallelism）与 Ring Attention

当序列非常长（如 128K, 1M tokens）时，单张 GPU 无法容纳完整的注意力矩阵（O(S²) 显存）。
上下文并行通过沿序列维度切分注意力计算来解决这个问题。

**Ring Attention 核心思想**:
1. 将序列沿长度切分给各 GPU
2. 每个 GPU 计算本地 Q @ 本地 KV → partial attention
3. 将 KV block 沿环形拓扑传递给下一个 GPU
4. 重复直到 KV 绕行一圈
5. 在线 softmax 累积: 每步使用 running max 和 running sum 更新

```
  GPU 0: Q0 @ [K0, K1, K2, K3]   ← 环形传递 KV
  GPU 1: Q1 @ [K1, K2, K3, K0]
  GPU 2: Q2 @ [K2, K3, K0, K1]
  GPU 3: Q3 @ [K3, K0, K1, K2]

  通信量: O(N × P) 而非 O(N²) — 长序列场景的关键优势!
```

In [ ]:
# Ring Attention 概念演示
print("=" * 50)
print("Ring Attention 演示")
print("=" * 50)

B, n_heads, seq_len, d_head = 1, 2, 16, 4
q = torch.randn(B, n_heads, seq_len, d_head)
k = torch.randn(B, n_heads, seq_len, d_head)
v = torch.randn(B, n_heads, seq_len, d_head)

# 模拟环形传递 (4 步)
num_steps = 4
print(f"\n序列长度: {seq_len}, 每个 head 维度: {d_head}")
print(f"环步数: {num_steps} (每个 step 计算 1/4 的 KV)")

outputs = []
for step in range(num_steps):
    output = ring_attention_step(q, k, v, step)
    outputs.append(output)
    print(f"  Step {step}: output shape {list(output.shape)}")

# 显存对比
print(f"\n注意力显存对比:")
total_tokens = seq_len
attn_matrix_size = total_tokens * total_tokens  # 简化
print(f"  完整注意力:      O(S²) = {seq_len}² = {attn_matrix_size} 个元素")
print(f"  Ring Attention:   O(S/P) per step = {seq_len}/{num_steps} = {seq_len//num_steps} 个元素")
print(f"  显存节省:         ≈ {(1 - 1/num_steps)*100:.0f}% (注意力矩阵部分)")

# 通信分析
print(f"\nRing Attention 通信分析:")
print(f"  每步传输: 1 份 KV block (大小 = B × n_heads × (S/P) × d_head)")
print(f"  总通信量: P × KV_block = B × n_heads × S × d_head (与完整 KV 相同)")
print(f"  对比朴素 All-to-All: Ring 通信量不变，但延迟更可控")

## 3. 上下文并行下的 Causal Mask

当序列被切分到多 GPU 后，因果注意力掩码需要相应调整：
- Rank 0: 标准下三角掩码（看不到任何其他 rank 的 token，因为是第一个分片）
- Rank k (>0): 需要看到前 k 个分片的所有 token + 自己分片内的 causal 约束

这被称为"扩展因果掩码"（extended causal mask），是实现正确性的关键。

In [ ]:
# CP Causal Mask 演示
print("=" * 50)
print("上下文并行 Causal Mask 演示")
print("=" * 50)

seq_len = 8
world_size = 2

print(f"\n序列长度: {seq_len}, CP size: {world_size}")
print(f"每个 rank 处理 {seq_len // world_size} 个 token")

for rank in range(world_size):
    mask = create_cp_causal_mask(seq_len=seq_len, rank=rank, world_size=world_size)
    chunk_size = seq_len // world_size
    local_start = rank * chunk_size
    
    print(f"\nRank {rank} causal mask (local tokens {local_start}:{local_start+chunk_size}):")
    print(f"  Mask 形状: {list(mask.shape)}")
    print(f"  可见 token 数 (False): {mask.numel() - mask.sum().item()}")
    print(f"  被掩盖 token 数 (True): {mask.sum().item()}")
    print(f"  可见范围: token 0 到 token {local_start + chunk_size - 1} (符合 causal)")

# 可视化: 哪个 rank 能看到哪些 token
print(f"\n序列切分可视化:")
for rank in range(world_size):
    chunk_size = seq_len // world_size
    local_start = rank * chunk_size
    visible = list(range(0, local_start + chunk_size))
    print(f"  Rank {rank} (持有 [{local_start}:{local_start+chunk_size})): 可看到 tokens {visible}")

# 不均分场景
print(f"\n非均匀切分 (seq_len=10, world_size=3):")
x_uneven = torch.randn(2, 10, 8)
for rank in range(3):
    chunk = partition_sequence(x_uneven, rank, 3)
    print(f"  Rank {rank}: chunk 形状 {list(chunk.shape)}")

## 4. CP + TP 混合策略显存分析

在大模型 + 长序列场景下，单独的 TP 或 CP 可能不够，需要组合使用。
TP 切分权重维度，CP 切分序列维度，两者互补：
- TP 减少每卡的模型权重显存
- CP 减少每卡的激活显存（序列维度）

总显存节省: ≈ 1 / (tp_size × cp_size)（理论值）

In [ ]:
# CP+TP 混合显存分析
print("=" * 50)
print("CP+TP 混合策略显存分析")
print("=" * 50)

configs = [
    (2048, 1024, 32, 2, 1, "标准 Llama 7B"),
    (4096, 2048, 32, 4, 2, "Llama 13B"),
    (8192, 4096, 64, 4, 4, "Llama 70B"),
    (16384, 4096, 64, 8, 4, "长序列大模型"),
    (32768, 5120, 128, 8, 8, "超长上下文"),
    (65536, 8192, 128, 8, 8, "128K 上下文"),
]

print(f"{'场景':<20s} {'seq':>6s} {'dim':>5s} {'TP':>3s} {'CP':>3s} {'单卡激活':>12s} {'节省比':>8s}")
print("-" * 70)

for label, seq_len, dim, n_heads, tp, cp in configs:
    mem = analyze_cp_tp_memory(seq_len, dim, n_heads, tp, cp)
    print(f"{label:<20s} {seq_len:>6d} {dim:>5d} {tp:>3d} {cp:>3d} "
          f"{mem['per_device_activation']:>10.1f}  {mem['reduction_ratio']:>7.1%}")

print(f"\n★ TP 切分模型权重, CP 切分激活序列 → 两者正交互补")
print(f"★ 总 GPU 数 = tp_size × cp_size × dp_size × pp_size")

# 并行策略推荐
print(f"\n" + "=" * 50)
print("并行策略推荐")
print("=" * 50)

scenarios = [
    (0.5, 2048, 4, "小模型 + 短序列"),
    (7.0, 4096, 8, "Llama 7B 风格"),
    (13.0, 8192, 16, "Llama 13B 风格"),
    (30.0, 16384, 32, "中等大模型 + 长序列"),
    (70.0, 32768, 64, "大模型 + 长序列"),
    (150.0, 65536, 128, "超大模型 + 超长序列"),
]

for model_gb, seq_len, n_gpus, desc in scenarios:
    config = recommend_parallel_config(model_gb, seq_len, n_gpus)
    print(f"  {desc:<25s}: {config}")

## 6. 相关文档

本 notebook 对应的详细原理文档：

- [专家并行详解](../../docs/parallel/expert-parallel.md) — Expert 分配、Token Dispatch、All-to-All 的完整原理
- [上下文并行详解](../../docs/parallel/context-parallel.md) — Ring Attention、Online Softmax、因果掩码调整
- [DeepSeek V3 架构详解](../../docs/models/deepseek-v3.md) — MoE 架构的前置知识

建议阅读顺序：先运行本 notebook 建立直觉，再阅读文档理解 Ring Attention 的在线 softmax 实现。

## 6. 练习与思考

1. **Top-K 路由**: 修改 Top-K 从 2 改为 1 或 3，观察 token 分布变化
2. **Ring Attention 显存**: 计算 seq_len=128K, world_size=8 时 Ring Attention vs 标准 Attention 的显存差异
3. **因果掩码验证**: 绘制 CP=4 时各 rank 的因果掩码矩阵，验证正确性

## 5. 关键要点总结

1. **专家并行 (EP)**: 将 MoE 的 Expert 分布到多 GPU，通过 All-to-All 通信实现 token dispatch。核心挑战是负载均衡。
2. **Token Dispatch**: Router 本地计算 → All-to-All 分发 token → 本地 Expert 计算 → All-to-All 回收结果
3. **上下文并行 (CP)**: 沿序列维度切分注意力计算，Ring Attention 将 O(S²) 显存降为 O(S/P)
4. **Ring Attention**: KV block 在环形拓扑中轮转，每步计算 partial attention + 在线 softmax 累积
5. **CP Causal Mask**: 需要扩展因果掩码，每个 rank 可见自己及之前所有分片的 token
6. **CP+TP 组合**: 两者正交互补 —— TP 减少权重显存，CP 减少激活显存
7. **推荐规则**: 模型 > 10GB 用 TP+PP+DP；序列 > 8K 加 CP；MoE 模型加 EP